In [ ]:
# ===== plan_d_final — CELL 0 : setup + সব space =====
# এটাই একমাত্র notebook যেটা submission বানায়। বাকি তিনটে (c / d / e) শুধু
# .npy score বানায়; এখানে সব cache-first-এ নিজে থেকে ঢুকে পড়ে।
#
# Accelerator: GPU।  Internet: OFF রাখলেও চলে যদি embedding attach করা থাকে।
import os, glob, json, time, re, gc, warnings
from collections import Counter
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

OUT, INP = '/kaggle/working', '/kaggle/input'
LABELS = ['same_figure','same_paper','related_papers','unrelated_papers']
COMBOS = ['image+text','text+text','image+image']
T0 = time.time()
def find(n, isdir=False):
    for r in (INP, OUT):
        for h in glob.glob(f'{r}/**/{n}', recursive=True):
            if os.path.isdir(h) == isdir: return h
    return None
def tlog(*a): print(f'[{(time.time()-T0)/60:6.1f} min]', *a, flush=True)
NCPU = os.cpu_count() or 4

ESS  = os.path.dirname(find('meta_train.parquet'))
IMGD = find('img384', isdir=True)
tlog('essentials:', ESS)

def nrm(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)
def center(x):
    x = np.asarray(x, dtype=np.float32)
    return nrm(x - x.mean(0, keepdims=True))

mtr = pd.read_parquet(f'{ESS}/meta_train.parquet')
mte = pd.read_parquet(f'{ESS}/meta_test.parquet')
for d in (mtr, mte):
    for c in ['t1','t2','h1','h2']: d[c] = d[c].astype(str)
    d['combo'] = np.where(d.t1 < d.t2, d.t1+'+'+d.t2, d.t2+'+'+d.t1)
mtr['y'] = mtr['label'].astype(str)

IH = pd.read_parquet(f'{ESS}/img_hashes.parquet').hash.astype(str).values
TH = pd.read_parquet(f'{ESS}/txt_hashes.parquet').hash.astype(str).values
II = {h:i for i,h in enumerate(IH)}; TI = {h:i for i,h in enumerate(TH)}
texts = pd.read_parquet(f'{ESS}/texts.parquet'); texts['hash'] = texts['hash'].astype(str)
TXT = texts.set_index('hash').loc[TH, 'text'].fillna('').astype(str).values

p = find('ocr_384x2.parquet') or find('ocr.parquet')
if p:
    o = pd.read_parquet(p); om = dict(zip(o.hash.astype(str), o.ocr.fillna('').astype(str)))
    OCR = np.array([om.get(h,'') for h in IH], dtype=object); OCR_OK = True
    tlog('OCR:', p)
else:
    OCR = np.array(['']*len(IH), dtype=object); OCR_OK = False
    print('⚠️ OCR নেই')

EMB = {'clip_i': nrm(np.load(f'{ESS}/emb_clip_img.npy')),
       'clip_t': nrm(np.load(f'{ESS}/emb_clip_txt.npy')),
       'sci_t' : center(np.load(f'{ESS}/emb_sci_txt.npy')),
       'dino_i': center(np.load(f'{ESS}/emb_dino_img.npy'))}
# Qwen3-VL 2B (প্রমাণিত: image+text 0.42 → 0.60) — আগের run-এর output attach করো
for key, fn in [('qvl_i','emb_qvl_img.npy'), ('qvl_t','emb_qvl_txt.npy'),
                ('qvl8_i','emb_qvl8_img.npy'), ('qvl8_t','emb_qvl8_txt.npy'),
                ('qvl_o','emb_qvl_ocr.npy')]:
    q = find(fn)
    if q: EMB[key] = nrm(np.load(q)); tlog('space', key, EMB[key].shape)
if 'qvl_t' not in EMB:
    print('🚨 qvl embedding পাওয়া যায়নি — plan_d_a-র output attach করো, নাহলে plan_c-র সমান ফল')

ALL = pd.concat([mtr.assign(split='tr'), mte.assign(split='te')], ignore_index=True)
NT, NI = len(TH), len(IH)
tlog('train', mtr.shape, 'test', mte.shape, '| spaces', sorted(EMB))


In [ ]:
# ===== plan_d_final — CELL 1 : বাইরের score সংগ্রহ + যাচাই =====
# plan_d_b → rr_train.npy / rr_test.npy            (image+text)
# plan_d_c → rr_train_image_image.npy ইত্যাদি      (বাকি দুই subset)
# plan_d_d → ce_oof.npy / ce_test.npy              (text cross-encoder, 10000×4)
# plan_d_e → vce_part0.npz + vce_part1.npz         (VL cross-encoder, দুই অর্ধ)
RR, EXT = {}, {}
NTR = {cb: int((mtr.combo == cb).sum()) for cb in COMBOS}
NTE = {cb: int((mte.combo == cb).sum()) for cb in COMBOS}

for cb in COMBOS:
    tg = cb.replace('+','_')
    cands = [(f'rr_train_{tg}.npy', f'rr_test_{tg}.npy')]
    if cb == 'image+text': cands.append(('rr_train.npy','rr_test.npy'))
    for fa, fb in cands:
        pa, pb = find(fa), find(fb)
        if not (pa and pb): continue
        a, b = np.load(pa), np.load(pb)
        # ⚠️ অসম্পূর্ণ scoring গড় দিয়ে ভরাট হয় — তখন train-এ প্রবল, test-এ ধ্রুবক,
        #    আর tree ওটার উপর ভরসা করে test-এ ধসে পড়ে। তাই কড়া যাচাই।
        ok = (len(a) == NTR[cb] and len(b) == NTE[cb]
              and len(np.unique(b[-200:])) > 20 and len(np.unique(a[-200:])) > 20)
        if ok:
            RR[cb] = np.concatenate([a, b]).astype(np.float32)
            tlog(f'rr[{cb}] ✅ {fa}')
        else:
            print(f'⚠️ rr[{cb}] বাদ — {fa}: len {len(a)}/{NTR[cb]}, {len(b)}/{NTE[cb]}, '
                  f'শেষ ২০০-তে অনন্য মান {len(np.unique(b[-200:]))} (অসম্পূর্ণ scoring?)')
        break

# plan_d_c2 → rr2_*  (proxy caption-এর উপর text reranker) — rr-এর পাশাপাশি, বদলে নয়
RR2 = {}
for cb in COMBOS:
    tg = cb.replace('+','_')
    pa, pb = find(f'rr2_train_{tg}.npy'), find(f'rr2_test_{tg}.npy')
    if not (pa and pb): continue
    a, b = np.load(pa), np.load(pb)
    if (len(a) == NTR[cb] and len(b) == NTE[cb]
            and len(np.unique(b[-200:])) > 20 and len(np.unique(a[-200:])) > 20):
        RR2[cb] = np.concatenate([a, b]).astype(np.float32); tlog(f'rr2[{cb}] ✅')
    else:
        print(f'⚠️ rr2[{cb}] বাদ — অসম্পূর্ণ scoring')

po, pt = find('ce_oof.npy'), find('ce_test.npy')
if po and pt:
    a, b = np.load(po), np.load(pt)
    if a.shape == (len(mtr),4) and b.shape == (len(mte),4):
        EXT['ce'] = (a.astype(np.float32), b.astype(np.float32)); tlog('ce ✅', a.shape)
    else: print('⚠️ ce আকার ভুল', a.shape, b.shape)

parts = [find(f'vce_part{f}.npz') for f in (0,1)]
parts = [np.load(q, allow_pickle=True) for q in parts if q]
if parts:
    vo = np.zeros((len(mtr),4), np.float32); seen = np.zeros(len(mtr), bool)
    vt = np.zeros((len(mte),4), np.float32)
    for z in parts:
        vo[z['idx']] = z['oof']; seen[z['idx']] = True; vt += z['test']/len(parts)
    cov = seen.mean()
    if cov > 0.99:
        EXT['vce'] = (vo, vt); tlog(f'vce ✅ {len(parts)} part, coverage {cov:.2%}')
    else:
        # এক অর্ধ এলে বাকি অর্ধে OOF নেই — feature-টা তখন অসৎ, তাই বাদ
        print(f'⚠️ vce বাদ — coverage মাত্র {cov:.2%}, দুটো part-ই লাগবে')

print('\nপাওয়া গেল → rr:', sorted(RR), '| rr2:', sorted(RR2), '| ext:', sorted(EXT))
if not RR and not RR2 and not EXT:
    print('কিছুই নেই — plan_d_a-র মূল ফলই আবার আসবে (pooled ~0.629)')


In [ ]:
# ===== plan_d_final — CELL 2 : feature engine (plan_d_a-র প্রমাণিত অংশ অপরিবর্তিত) =====
from sklearn.feature_extraction.text import TfidfVectorizer

def topk_mean(Q, P, k, bs=2048):
    out = np.zeros(len(Q), np.float32)
    for i in range(0, len(Q), bs):
        S = Q[i:i+bs] @ P.T
        out[i:i+bs] = np.partition(S, -k, axis=1)[:, -k:].mean(1)
    return out

def rank_in_pool(Q, P, iq, ip, bs=2048):
    rk = np.zeros(len(iq), np.float32)
    for i in range(0, len(iq), bs):
        S = Q[iq[i:i+bs]] @ P.T
        s = S[np.arange(S.shape[0]), ip[i:i+bs]]
        rk[i:i+bs] = (S > s[:, None]).sum(1)
    return np.log1p(rk)

def best_match(Q, P, rP, k=5, bs=2048):
    out = np.zeros((len(Q), k), np.int64)
    for i in range(0, len(Q), bs):
        S = 2*(Q[i:i+bs] @ P.T) - rP[None, :]
        out[i:i+bs] = np.argsort(-S, 1)[:, :k]
    return out

RAD = {}
def radius(a_, b_):
    if (a_, b_) not in RAD:
        RAD[(a_, b_)] = topk_mean(EMB[a_], EMB[b_], 11 if a_ == b_ else 10)
    return RAD[(a_, b_)]

XT, XI = ('qvl_t','qvl_i') if 'qvl_t' in EMB else ('clip_t','clip_i')
T2I = best_match(EMB[XT], EMB[XI], radius(XI, XT))
I2T = best_match(EMB[XI], EMB[XT], radius(XT, XI))
tlog('2-hop proxy:', XT, XI)

DOCS = np.concatenate([TXT, OCR]).astype(str)          # text j → j ; image i → NT+i
LW = TfidfVectorizer(token_pattern=r'(?u)\b[\w\-\.\+]*\w\b', lowercase=False,
                     sublinear_tf=True, max_df=0.5).fit_transform(DOCS)
LC = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), sublinear_tf=True,
                     min_df=2, max_features=400000).fit_transform(DOCS)
_tok = re.compile(r'[A-Za-z0-9][\w\-\.\+]*[A-Za-z0-9]|\d')
TOKS = [set(t for t in _tok.findall(s) if len(t) >= 2 or t.isdigit()) for s in DOCS]
DF = Counter(t for s in TOKS[:NT] for t in s)
_idf = lambda t: np.log((NT+1)/(DF.get(t,0)+1))
RARE  = [{t for t in s if DF.get(t,0) <= 10 and len(t) >= 3} for s in TOKS]
ENT   = [{t for t in s if (any(c.isdigit() for c in t) and any(c.isalpha() for c in t))
          or (t.isupper() and len(t) >= 2)} for s in TOKS]
NUM   = [{t for t in s if re.fullmatch(r'\d+(\.\d+)?', t) and len(t) >= 2} for s in TOKS]
LOWER = [{t.lower() for t in s if len(t) >= 4 and t.isalpha()} for s in TOKS]
tlog('lexical index ready')

def lex(da, db, p):
    F = {f'{p}_tfw': np.asarray(LW[da].multiply(LW[db]).sum(1)).ravel(),
         f'{p}_tfc': np.asarray(LC[da].multiply(LC[db]).sum(1)).ravel()}
    rn, rj, en, nn_, ws, wj, lw = ([] for _ in range(7))
    for x, y2 in zip(da, db):
        A_, B_ = RARE[x], RARE[y2]; I = A_ & B_
        rn.append(len(I)); rj.append(len(I)/max(1, len(A_|B_)))
        en.append(len(ENT[x] & ENT[y2])); nn_.append(len(NUM[x] & NUM[y2]))
        TA, TB = TOKS[x], TOKS[y2]; I2 = TA & TB
        si = sum(_idf(t) for t in I2); su = sum(_idf(t) for t in TA|TB)
        ws.append(si); wj.append(si/max(1e-6, su)); lw.append(len(LOWER[x] & LOWER[y2]))
    F.update({f'{p}_rare_n':rn, f'{p}_rare_j':rj, f'{p}_ent_n':en, f'{p}_num_n':nn_,
              f'{p}_idf_sum':ws, f'{p}_idf_j':wj, f'{p}_word_n':lw})
    return F

def sim_bundle(F, name, Ea, Eb, ia, ib, same_mod):
    A_, B_ = EMB[Ea], EMB[Eb]
    c = (A_[ia] * B_[ib]).sum(1)
    ra, rb = radius(Ea, Eb)[ia], radius(Eb, Ea)[ib]
    F[f'{name}_cos'] = c
    F[f'{name}_csls'] = 2*c - ra - rb
    k1 = rank_in_pool(A_, B_, ia, ib); k2 = rank_in_pool(B_, A_, ib, ia)
    if same_mod:
        F[f'{name}_rmin'], F[f'{name}_rmax'] = np.minimum(k1,k2), np.maximum(k1,k2)
        F[f'{name}_hmin'], F[f'{name}_hmax'] = np.minimum(ra,rb), np.maximum(ra,rb)
    else:
        F[f'{name}_r12'], F[f'{name}_r21'] = k1, k2
        F[f'{name}_h1'],  F[f'{name}_h2']  = ra, rb

def pair_index(D):
    cb = D.combo.iloc[0]
    if cb == 'image+text':
        sw = (D.t1 == 'image').values
        a_ = np.array([TI[h] for h in np.where(sw, D.h2, D.h1)])
        b_ = np.array([II[h] for h in np.where(sw, D.h1, D.h2)])
        la, lb = np.where(sw, D.len2, D.len1), np.where(sw, D.len1, D.len2)
        oa, ob = a_, NT + b_
    elif cb == 'text+text':
        a_ = np.array([TI[h] for h in D.h1]); b_ = np.array([TI[h] for h in D.h2])
        la, lb = D.len1.values, D.len2.values; oa, ob = a_, b_
    else:
        a_ = np.array([II[h] for h in D.h1]); b_ = np.array([II[h] for h in D.h2])
        la, lb = D.len1.values, D.len2.values; oa, ob = NT + a_, NT + b_
    return a_, b_, np.log1p(la.astype(float)), np.log1p(lb.astype(float)), oa, ob

def sym(F, n, x, y): F[f'{n}_min'], F[f'{n}_max'] = np.minimum(x,y), np.maximum(x,y)


In [ ]:
# ===== plan_d_final — CELL 3 : বাইরের score → feature =====
# rr একটা column হিসেবে দিলে tree ওটাকে নিরঙ্কুশ মান হিসেবে দেখে। কিন্তু asল তথ্য
# আপেক্ষিক: এই caption-এর অন্য figure-গুলোর তুলনায় এটা কত ভালো। test-এ 1,707 row-এর
# caption একাধিক জোড়ায় আসে, তাই এই বিয়োগটাই আসল বৈষম্য তৈরি করে।
def rr_bundle(F, r, D, oa, ob, p):
    assert len(r) == len(D), (p, len(r), len(D))
    F[p] = r
    F[f'{p}_rank'] = pd.Series(r).rank(pct=True).values.astype(np.float32)
    obj = np.concatenate([oa, ob]); val = np.concatenate([r, r])
    m = pd.Series(val).groupby(obj).mean()
    da = r - m.reindex(oa).values.astype(np.float32)
    db = r - m.reindex(ob).values.astype(np.float32)
    sym(F, f'{p}_d', da, db)                         # swap-নিরপেক্ষ
    cnt = pd.Series(np.ones(len(obj))).groupby(obj).sum()
    sym(F, f'{p}_deg', np.log1p(cnt.reindex(oa).values), np.log1p(cnt.reindex(ob).values))

def ext_block(F, cb, D, oa, ob):
    if cb in RR:  rr_bundle(F, RR[cb],  D, oa, ob, 'rr')
    if cb in RR2: rr_bundle(F, RR2[cb], D, oa, ob, 'rr2')
    if cb in RR and cb in RR2:
        # দুটো reranker আলাদা জিনিস দেখে (ছবি বনাম proxy caption) — তাদের মতানৈক্যই তথ্য
        F['rr_gap'] = (pd.Series(RR[cb]).rank(pct=True).values
                       - pd.Series(RR2[cb]).rank(pct=True).values).astype(np.float32)
    for nm in ('ce','vce'):
        if nm not in EXT: continue
        a, b = EXT[nm]
        v = np.vstack([a[(mtr.combo == cb).values], b[(mte.combo == cb).values]]).astype(np.float32)
        assert len(v) == len(D), (nm, cb, len(v), len(D))
        for j, c in enumerate(LABELS): F[f'{nm}_{c}'] = v[:, j]
        s = np.sort(v, 1)
        F[f'{nm}_margin'] = s[:, -1] - s[:, -2]
        F[f'{nm}_ent'] = -(v*np.log(v.clip(1e-9))).sum(1)
        # যে দুটো সীমানায় আমরা হারছি, সেগুলোর অনুপাত সরাসরি দিয়ে দিই
        F[f'{nm}_sp_rel'] = v[:,1]/(v[:,1]+v[:,2]+1e-9)
        F[f'{nm}_rel_unrel'] = v[:,2]/(v[:,2]+v[:,3]+1e-9)

def build_scalar(D):
    cb = D.combo.iloc[0]; a_, b_, la, lb, oa, ob = pair_index(D); F = {}
    if cb == 'image+text':
        for ts, isp in [('clip_t','clip_i'), ('qvl_t','qvl_i'), ('qvl8_t','qvl8_i')]:
            if ts in EMB and isp in EMB: sim_bundle(F, ts[:-2], ts, isp, a_, b_, False)
        if OCR_OK:
            F.update(lex(a_, NT + b_, 'cap_ocr'))
            F['ocr_len'] = np.log1p([len(OCR[i]) for i in b_])
        F['px_dino']  = (EMB['dino_i'][T2I[a_,0]] * EMB['dino_i'][b_]).sum(1)
        F['px_dino5'] = (nrm(EMB['dino_i'][T2I[a_]].mean(1)) * EMB['dino_i'][b_]).sum(1)
        F['px_sci']   = (EMB['sci_t'][I2T[b_,0]] * EMB['sci_t'][a_]).sum(1)
        F['px_sci5']  = (nrm(EMB['sci_t'][I2T[b_]].mean(1)) * EMB['sci_t'][a_]).sum(1)
        F.update(lex(I2T[b_,0], a_, 'px_cap'))
        F['self_t'] = (T2I[a_] == b_[:,None]).any(1).astype(np.float32)
        F['self_i'] = (I2T[b_] == a_[:,None]).any(1).astype(np.float32)
        F['len_t'], F['len_i'] = la, lb
    elif cb == 'text+text':
        for sp in ['clip_t','sci_t','qvl_t','qvl8_t']:
            if sp in EMB: sim_bundle(F, sp, sp, sp, a_, b_, True)
        F.update(lex(a_, b_, 'cap'))
        F['px_dino']  = (EMB['dino_i'][T2I[a_,0]] * EMB['dino_i'][T2I[b_,0]]).sum(1)
        F['px_dino5'] = (nrm(EMB['dino_i'][T2I[a_]].mean(1)) * nrm(EMB['dino_i'][T2I[b_]].mean(1))).sum(1)
        sym(F, 'len', la, lb)
    else:
        for sp in ['clip_i','dino_i','qvl_i','qvl8_i']:
            if sp in EMB: sim_bundle(F, sp, sp, sp, a_, b_, True)
        if 'qvl_o' in EMB:                     # OCR-এর semantic space — image+image সবচেয়ে দুর্বল
            sim_bundle(F, 'qvl_ocr', 'qvl_o', 'qvl_o', a_, b_, True)
        if OCR_OK:
            F.update(lex(NT + a_, NT + b_, 'ocr'))
            sym(F, 'ocr_len', np.log1p([len(OCR[i]) for i in a_]), np.log1p([len(OCR[i]) for i in b_]))
        F['px_sci']  = (EMB['sci_t'][I2T[a_,0]] * EMB['sci_t'][I2T[b_,0]]).sum(1)
        F['px_sci5'] = (nrm(EMB['sci_t'][I2T[a_]].mean(1)) * nrm(EMB['sci_t'][I2T[b_]].mean(1))).sum(1)
        F.update(lex(I2T[a_,0], I2T[b_,0], 'px_cap'))
        sym(F, 'len', la, lb)
    ext_block(F, cb, D, oa, ob)
    return pd.DataFrame({k: np.asarray(v, dtype=np.float32) for k, v in F.items()})

FULL = {'image+text':  [('clip_t','clip_i'), ('qvl_t','qvl_i'), ('qvl8_t','qvl8_i')],
        'text+text':   [('clip_t','clip_t'), ('sci_t','sci_t'), ('qvl_t','qvl_t'), ('qvl8_t','qvl8_t')],
        'image+image': [('clip_i','clip_i'), ('dino_i','dino_i'), ('qvl_i','qvl_i'), ('qvl8_i','qvl8_i')]}

def build_full(D):
    a_, b_, _, _, _, _ = pair_index(D); B = []
    for sa, sb in FULL[D.combo.iloc[0]]:
        if sa in EMB and sb in EMB:
            e1, e2 = EMB[sa][a_], EMB[sb][b_]
            B += [np.abs(e1-e2), e1*e2]
    return np.hstack(B).astype(np.float32)

DATA = {}
for cb in COMBOS:
    D = ALL[ALL.combo == cb].reset_index(drop=True)
    S, Fu = build_scalar(D), build_full(D)
    assert np.isfinite(S.values).all() and np.isfinite(Fu).all()
    DATA[cb] = (D, S, Fu)
    tlog(f'{cb}: rows={len(D)} scalar={S.shape[1]} full={Fu.shape[1]}')


In [ ]:
# ===== plan_d_final — CELL 4 : train → submission =====
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# plan_d_a-র মাপা ফল — এর নিচে নামলে কিছু ভেঙেছে
REF = {'image+text': 0.6017, 'text+text': 0.6725, 'image+image': 0.5135}
REF_POOLED = 0.6293
REF_AUC = {'image+text': (0.722, 0.768), 'text+text': (0.797, 0.846), 'image+image': (0.672, 0.683)}
FOLDS = StratifiedKFold(5, shuffle=True, random_state=0)      # c/d/e-র সাথে হুবহু এক

def lgb_p(K, **kw):
    p = dict(objective='multiclass', num_class=K, n_estimators=700, learning_rate=0.05,
             num_leaves=31, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
             class_weight='balanced', min_child_samples=20, reg_lambda=1.0,
             max_bin=63, force_col_wise=True, verbose=-1, n_jobs=NCPU)
    p.update(kw); return p

def run_lgb(X, y, Xt, K, seeds=(0,), **kw):
    oof = np.zeros((len(y),K)); te = np.zeros((len(Xt),K))
    for s in seeds:
        for tr_, va in FOLDS.split(X, y):
            m = lgb.LGBMClassifier(**lgb_p(K, random_state=s, **kw)).fit(X[tr_], y[tr_])
            oof[va] += m.predict_proba(X[va])/len(seeds); te += m.predict_proba(Xt)/(5*len(seeds))
    return oof, te

def run_lr(X, y, Xt, K, C=0.05):
    oof = np.zeros((len(y),K)); te = np.zeros((len(Xt),K))
    for tr_, va in FOLDS.split(X, y):
        sc = StandardScaler().fit(X[tr_])
        m = LogisticRegression(C=C, max_iter=300, class_weight='balanced').fit(sc.transform(X[tr_]), y[tr_])
        oof[va] = m.predict_proba(sc.transform(X[va])); te += m.predict_proba(sc.transform(Xt))/5
    return oof, te

mf1 = lambda y, P, w=None: f1_score(y, (P*(1 if w is None else w)).argmax(1), average='macro')

def blend_w(oofs, y, step=0.1):
    names = list(oofs); best = (-1, None); grid = np.arange(0, 1+1e-9, step)
    def rec(i, left, cur):
        nonlocal best
        if i == len(names)-1:
            w = cur + [left]
            s = mf1(y, sum(wi*oofs[n] for wi, n in zip(w, names)))
            if s > best[0]: best = (s, {n: round(float(x),2) for n, x in zip(names, w)})
            return
        for g in grid:
            if g <= left+1e-9: rec(i+1, left-g, cur+[g])
    rec(0, 1.0, []); return best

def class_mult(P, y, rounds=10):
    # ⚠️ IPF/Sinkhorn দিয়ে ঠিক 1000 করে চাপানোর চেষ্টা মাপা হয়েছে: OOF-এ −0.0087।
    #    macro-F1-এর সর্বোচ্চ বিন্দু ভারসাম্যপূর্ণ গণনায় নয়। তাই গুণকই থাক।
    w = np.ones(P.shape[1]); best = mf1(y, P, w)
    for _ in range(rounds):
        moved = False
        for j in range(P.shape[1]):
            for m in (0.7,0.8,0.9,0.95,1.05,1.1,1.25,1.4):
                w2 = w.copy(); w2[j] *= m
                s = mf1(y, P, w2)
                if s > best+1e-5: best, w, moved = s, w2, True
        if not moved: break
    return w, best

sub = pd.DataFrame({'id': mte.id.values})
for c in LABELS: sub[c] = 0
REPORT, SIZES, POOL = {}, {}, {'y': [], 'p': []}

for cb in COMBOS:
    D, S, Fu = DATA[cb]
    m_ = (D.split == 'tr').values
    d, dt = D[m_].reset_index(drop=True), D[~m_].reset_index(drop=True)
    classes = [c for c in LABELS if c in set(d.y)]; K = len(classes)
    y = d.y.map({c:i for i,c in enumerate(classes)}).values
    Xs, Xst = S.values[m_], S.values[~m_]
    Xa, Xat = np.hstack([Xs, Fu[m_]]), np.hstack([Xst, Fu[~m_]])
    tlog(f'--- {cb}: n={len(y)} K={K} scalar={Xs.shape[1]} all={Xa.shape[1]}')

    oofs, tests = {}, {}
    oofs['lgb_s'], tests['lgb_s'] = run_lgb(Xs, y, Xst, K, seeds=(0,1,2), n_estimators=600,
                                            learning_rate=0.03, num_leaves=15, colsample_bytree=0.7)
    tlog(f'  lgb_scalar {mf1(y, oofs["lgb_s"]):.4f}')
    oofs['lgb_a'], tests['lgb_a'] = run_lgb(Xa, y, Xat, K, seeds=(0,1))
    tlog(f'  lgb_all    {mf1(y, oofs["lgb_a"]):.4f}')
    oofs['lr_a'],  tests['lr_a']  = run_lr(Xa, y, Xat, K)
    tlog(f'  lr_all     {mf1(y, oofs["lr_a"]):.4f}')

    s_bl, wts = blend_w(oofs, y)
    P  = sum(wts[n]*oofs[n]  for n in oofs)
    Pt = sum(wts[n]*tests[n] for n in tests)
    mult, s_fin = class_mult(P, y)

    # cascade: related বনাম unrelated-এর জন্য আলাদা binary model (মাপা +0.001…+0.011)
    s_cas, mult_c, obt = s_fin, None, None
    if 'related_papers' in classes and 'unrelated_papers' in classes:
        iR, iU = classes.index('related_papers'), classes.index('unrelated_papers')
        msk = np.isin(y, [iR, iU]); posm = np.where(msk)[0]
        yb = (y[msk] == iR).astype(int)
        ob = np.zeros(msk.sum()); obt = np.zeros(len(Xat))
        for tri, vai in StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(msk.sum()), yb):
            bm = lgb.LGBMClassifier(**lgb_p(2, objective='binary', num_class=1)).fit(Xa[posm[tri]], yb[tri])
            ob[vai] = bm.predict_proba(Xa[posm[vai]])[:, 1]
            obt += bm.predict_proba(Xat)[:, 1] / 5
        BIN = np.full(len(y), np.nan); BIN[posm] = ob
        def cascade(Pm, B, w, th):
            pr = Pm.argmax(1).copy()
            amb = np.isin(pr, [iR, iU]) & ~np.isnan(B)
            r, u = Pm[amb, iR], Pm[amb, iU]
            pr[amb] = np.where((1-w)*(r/(r+u+1e-9)) + w*B[amb] > th, iR, iU)
            return pr
        Pm = P*mult; best = (s_fin, (0.0, 0.5))
        for w in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0):
            for th in (0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7):
                sc = f1_score(y, cascade(Pm, BIN, w, th), average='macro')
                if sc > best[0]: best = (sc, (w, th))
        s_cas, mult_c = best
        tlog(f'  cascade {s_fin:.4f} → {s_cas:.4f}  w={mult_c[0]} th={mult_c[1]}')

    # সৎ যাচাই — এলোমেলো permutation বাধ্যতামূলক (CSV label অনুযায়ী sorted)
    _rs = np.random.default_rng(0).permutation(len(y)); h = len(y)//2
    wh, _ = class_mult(P[_rs[:h]], y[_rs[:h]])
    honest = mf1(y[_rs[h:]], P[_rs[h:]], wh)

    # যে দুটো সীমানায় আমরা হারছি — এগুলোই আসল অগ্রগতির মাপ
    aucs = {}
    for a, b in [('same_paper','related_papers'), ('related_papers','unrelated_papers')]:
        if a not in classes or b not in classes: continue
        ia, ib = classes.index(a), classes.index(b)
        s = np.isin(y, [ia, ib]); t = (y[s] == ia).astype(int)
        aucs[f'{a[:8]}_vs_{b[:7]}'] = round(float(roc_auc_score(t, P[s,ia]/(P[s,ia]+P[s,ib]+1e-9))), 3)
    tlog(f'  blend {wts} → {s_bl:.4f} | +thr {s_fin:.4f} | +cascade {s_cas:.4f} | honest {honest:.4f}')
    tlog(f'  AUC {aucs}  (plan_d_a: {REF_AUC[cb]})   [plan_d_a F1 {REF[cb]:.4f}, পার্থক্য {s_cas-REF[cb]:+.4f}]')
    print(classification_report(y, (P*mult).argmax(1), target_names=classes, digits=3))

    REPORT[cb] = {**{n: round(mf1(y,o),4) for n,o in oofs.items()}, 'blend_w': wts,
                  'blend': round(s_bl,4), 'thr': round(s_fin,4), 'final': round(s_cas,4),
                  'honest': round(float(honest),4), 'auc': aucs, 'plan_d_a': REF[cb]}
    SIZES[cb] = len(y)
    tg = cb.replace('+','_')
    np.save(f'{OUT}/F_oof_{tg}.npy',  (P*mult).astype(np.float32))
    np.save(f'{OUT}/F_test_{tg}.npy', (Pt*mult).astype(np.float32))
    json.dump(classes, open(f'{OUT}/F_classes_{tg}.json','w'))

    Ptm = Pt*mult
    if mult_c is not None and s_cas > s_fin + 1e-4:
        prt = Ptm.argmax(1).copy(); amb = np.isin(prt, [iR, iU])
        r, u = Ptm[amb, iR], Ptm[amb, iU]; w, th = mult_c
        prt[amb] = np.where((1-w)*(r/(r+u+1e-9)) + w*obt[amb] > th, iR, iU)
        pred = prt
        oofpred = cascade(P*mult, BIN, *mult_c)
    else:
        pred = Ptm.argmax(1); oofpred = (P*mult).argmax(1)
    POOL['y'] += list(d.y.values); POOL['p'] += [classes[i] for i in oofpred]
    pos = pd.Index(sub.id).get_indexer(dt.id.values); assert (pos >= 0).all()
    for j, c in enumerate(classes): sub.loc[pos[pred == j], c] = 1

assert len(sub) == len(mte) and sub.id.is_unique
assert (sub[LABELS].sum(1) == 1).all()
sub[['id']+LABELS].to_csv(f'{OUT}/submission.csv', index=False)

# LB pooled macro-F1 মাপে, subset-গড় নয় — তাই এটাই আসল সংখ্যা
pooled = f1_score(POOL['y'], POOL['p'], average='macro')
tot = sum(SIZES.values())
REPORT['pooled_oof']   = round(float(pooled), 4)
REPORT['pooled_ref']   = REF_POOLED
REPORT['subset_mean']  = round(sum(REPORT[c]['final']*SIZES[c] for c in SIZES)/tot, 4)
REPORT['spaces']       = sorted(EMB)
REPORT['extra']        = {'rr': sorted(RR), 'ext': sorted(EXT)}
json.dump(REPORT, open(f'{OUT}/report_final.json','w'), indent=1, default=str)
print('\n'+'='*64); print(json.dumps(REPORT, indent=1, default=str))
print('\n' + classification_report(POOL['y'], POOL['p'], labels=LABELS, digits=3))
print(f'>>> POOLED OOF {pooled:.4f}   (plan_d_a ছিল {REF_POOLED:.4f}, পার্থক্য {pooled-REF_POOLED:+.4f})')
print('    LB সাধারণত এর ±0.01-এর মধ্যে থাকে (0.4806 → 0.48081 মিলেছিল)')
print('label counts:', sub[LABELS].sum().to_dict())
tlog('done →', f'{OUT}/submission.csv')
